In [1]:
from pathlib import Path
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv


env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)  # Load environment variables from workspace root .env file


llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
    # other params...
)


In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# ── State ────────────────────────────────────────────────────────────────────
class State(TypedDict):
    topic: str
    pros_analysis: str
    cons_analysis: str
    final_summary: str

# ── Node 1: Analyse Pros (runs in parallel) ───────────────────────────────────
def analyze_pros(state: State) -> dict:
    response = llm.invoke([
        SystemMessage(content="You are an expert analyst. List the top 3 PROS of the given topic concisely."),
        HumanMessage(content=f"Topic: {state['topic']}"),
    ])
    print("[analyze_pros] done")
    return {"pros_analysis": response.content}

# ── Node 2: Analyse Cons (runs in parallel) ───────────────────────────────────
def analyze_cons(state: State) -> dict:
    response = llm.invoke([
        SystemMessage(content="You are an expert analyst. List the top 3 CONS of the given topic concisely."),
        HumanMessage(content=f"Topic: {state['topic']}"),
    ])
    print("[analyze_cons] done")
    return {"cons_analysis": response.content}

# ── Node 3: Summarise (fan-in) ────────────────────────────────────────────────
def summarize(state: State) -> dict:
    response = llm.invoke([
        SystemMessage(content="You are a concise summarizer. Combine the pros and cons into one balanced paragraph."),
        HumanMessage(content=f"Pros:\n{state['pros_analysis']}\n\nCons:\n{state['cons_analysis']}"),
    ])
    print("[summarize] done")
    return {"final_summary": response.content}

# ── Build Graph ───────────────────────────────────────────────────────────────
builder = StateGraph(State)

builder.add_node("analyze_pros", analyze_pros)
builder.add_node("analyze_cons", analyze_cons)
builder.add_node("summarize",    summarize)

# Fan-out: START → both parallel nodes
builder.add_edge(START, "analyze_pros")
builder.add_edge(START, "analyze_cons")

# Fan-in: both parallel nodes → summarize → END
builder.add_edge("analyze_pros", "summarize")
builder.add_edge("analyze_cons", "summarize")
builder.add_edge("summarize",    END)

graph = builder.compile()
print("Graph compiled successfully.")


Graph compiled successfully.


In [3]:
# ── Run the parallel graph ────────────────────────────────────────────────────
initial_state = State(
    topic="Artificial Intelligence in Healthcare",
    pros_analysis="",
    cons_analysis="",
    final_summary="",
)

result = graph.invoke(initial_state)

print("=" * 60)
print("TOPIC:", initial_state["topic"])
print("=" * 60)
print("\n--- PROS (Node 1) ---")
print(result["pros_analysis"])
print("\n--- CONS (Node 2) ---")
print(result["cons_analysis"])
print("\n--- FINAL SUMMARY (fan-in node) ---")
print(result["final_summary"])


[analyze_pros] done
[analyze_cons] done
[summarize] done
TOPIC: Artificial Intelligence in Healthcare

--- PROS (Node 1) ---
1. **Enhanced Diagnostic Accuracy**: AI analyzes medical data (e.g., imaging, lab results) with high precision, enabling early detection of diseases like cancer or cardiovascular conditions.  
2. **Personalized Treatment Plans**: AI processes patient-specific data (genetics, lifestyle) to tailor therapies, improving outcomes and reducing adverse effects.  
3. **Operational Efficiency**: Automates administrative tasks (scheduling, documentation) and optimizes resource allocation, reducing costs and improving healthcare accessibility.

--- CONS (Node 2) ---
**Top 3 CONS of Artificial Intelligence in Healthcare:**  
1. **Privacy and Data Security Risks**: Sensitive patient data used to train AI systems may be vulnerable to breaches, misuse, or unauthorized access, raising ethical and legal concerns.  
2. **Algorithmic Bias and Inequality**: AI models trained on non-